# 🛩️ ISR Intelligence Dashboard - Iran/Gulf Region

**Airborne Swarm Deconfliction & Intelligence Analysis**

This notebook provides:
- Synthetic ISR track generation (MQ-9, RC-135, RQ-4, P-8)
- Fleet composition analytics
- Time on Station (ToS) calculations
- Interactive map visualization
- Data export (CSV, GeoJSON, KML)

**No ArcGIS Pro license required** - uses open-source tools.

---
Built by Cleo ✨ | March 2026

## 📦 1. Install Dependencies

In [ ]:
!pip install -q folium pandas numpy geopandas shapely simplekml plotly scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap, TimestampedGeoJson, AntPath
import geopandas as gpd
from shapely.geometry import Point, LineString
from datetime import datetime, timedelta
import json
import simplekml
from sklearn.cluster import DBSCAN
import plotly.express as px
import plotly.graph_objects as go
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print('✅ All dependencies loaded!')

## 🎯 2. Define Region of Interest

Centered on Iran and the Strait of Hormuz region.

In [ ]:
# Region bounds
REGION = {
    'name': 'Iran / Gulf Region',
    'center_lat': 32.0,
    'center_lon': 53.0,
    'lat_range': (24.0, 40.0),
    'lon_range': (44.0, 64.0),
}

# Key strategic locations
STRATEGIC_POINTS = {
    'Strait of Hormuz': (26.5667, 56.2500),
    'Tehran': (35.6892, 51.3890),
    'Bandar Abbas': (27.1832, 56.2666),
    'Isfahan (Nuclear)': (32.6546, 51.6680),
    'Natanz (Nuclear)': (33.7250, 51.7167),
    'Bushehr (Nuclear)': (28.9684, 50.8385),
    'Al Udeid (US)': (25.1173, 51.3150),
    'Al Dhafra (US)': (24.2481, 54.5476),
    'Bahrain NAVCENT': (26.2285, 50.5860),
}

print(f"📍 Region: {REGION['name']}")
print(f"📍 Strategic Points: {len(STRATEGIC_POINTS)}")

## ✈️ 3. Generate Synthetic ISR Tracks

Simulates realistic flight patterns for various ISR platforms:
- **MQ-9 Reaper**: Loitering patterns over targets
- **RC-135 Rivet Joint**: Wide orbit SIGINT collection
- **RQ-4 Global Hawk**: High-altitude transit/surveillance
- **P-8 Poseidon**: Maritime patrol patterns

In [ ]:
def generate_isr_tracks(num_points=100, seed=42):
    """Generate synthetic ADS-B style tracks for ISR platforms over Iran region."""
    np.random.seed(seed)
    start_time = datetime.utcnow()
    
    # Platform definitions: (type, tail, base_lat, base_lon, variance, altitude_range, pattern)
    platforms = [
        ('MQ-9', 'T101', 32.5, 53.5, 0.02, (18000, 25000), 'loiter'),      # Over central Iran
        ('MQ-9', 'T102', 27.0, 56.5, 0.015, (20000, 25000), 'loiter'),     # Strait of Hormuz
        ('RC-135', 'R201', 33.0, 52.0, 0.08, (28000, 35000), 'orbit'),     # Wide SIGINT orbit
        ('RC-135', 'R202', 28.5, 58.0, 0.06, (30000, 35000), 'orbit'),     # Gulf of Oman
        ('RQ-4', 'G301', 31.0, 55.0, 0.05, (55000, 60000), 'transit'),     # High altitude
        ('RQ-4', 'G302', 35.0, 50.0, 0.04, (55000, 60000), 'racetrack'),   # Northern Iran
        ('P-8', 'P401', 26.0, 57.0, 0.03, (15000, 25000), 'maritime'),     # Maritime patrol
        ('P-8', 'P402', 25.5, 54.0, 0.025, (18000, 28000), 'maritime'),    # Persian Gulf
    ]
    
    data = []
    
    for platform, tail, base_lat, base_lon, variance, alt_range, pattern in platforms:
        for i in range(num_points):
            # Generate position based on flight pattern
            t = i / 10.0
            
            if pattern == 'loiter':
                # Tight circular loiter
                lat = base_lat + variance * np.sin(t) + np.random.normal(0, 0.005)
                lon = base_lon + variance * np.cos(t) + np.random.normal(0, 0.005)
                heading = (np.degrees(np.arctan2(np.cos(t), -np.sin(t))) + 360) % 360
            elif pattern == 'orbit':
                # Wide elliptical orbit
                lat = base_lat + variance * 1.5 * np.sin(t * 0.7) + np.random.normal(0, 0.01)
                lon = base_lon + variance * np.cos(t * 0.7) + np.random.normal(0, 0.01)
                heading = (np.degrees(np.arctan2(np.cos(t * 0.7), -np.sin(t * 0.7) * 1.5)) + 360) % 360
            elif pattern == 'transit':
                # Linear transit with slight drift
                lat = base_lat + i * 0.02 + np.random.normal(0, 0.005)
                lon = base_lon + i * 0.015 + np.random.normal(0, 0.005)
                heading = 45 + np.random.normal(0, 5)
            elif pattern == 'racetrack':
                # Racetrack pattern
                phase = (i % 40) / 40.0 * 2 * np.pi
                if phase < np.pi:
                    lat = base_lat + variance * np.sin(phase)
                    lon = base_lon + 0.1 * (phase / np.pi)
                else:
                    lat = base_lat + variance * np.sin(phase)
                    lon = base_lon + 0.1 * (2 - phase / np.pi)
                heading = 90 if phase < np.pi else 270
            elif pattern == 'maritime':
                # Maritime search pattern (expanding square)
                leg = i // 10
                pos_in_leg = (i % 10) / 10.0
                directions = [(1, 0), (0, 1), (-1, 0), (0, -1)]
                d = directions[leg % 4]
                scale = (leg // 4 + 1) * variance
                lat = base_lat + d[0] * scale * pos_in_leg + np.random.normal(0, 0.003)
                lon = base_lon + d[1] * scale * pos_in_leg + np.random.normal(0, 0.003)
                heading = [90, 0, 270, 180][leg % 4]
            
            # Calculate altitude and speed
            altitude = np.random.randint(alt_range[0], alt_range[1])
            speed = {'MQ-9': 180, 'RC-135': 280, 'RQ-4': 340, 'P-8': 320}[platform]
            speed += np.random.randint(-20, 20)
            
            timestamp = start_time + timedelta(minutes=i * 2)
            
            data.append({
                'Timestamp': timestamp,
                'Platform': platform,
                'Tail_Number': tail,
                'Latitude': round(lat, 6),
                'Longitude': round(lon, 6),
                'Altitude_ft': altitude,
                'Heading': round(heading, 1),
                'Speed_kts': speed,
                'Pattern': pattern,
            })
    
    df = pd.DataFrame(data)
    return df

# Generate tracks
df = generate_isr_tracks(num_points=100)
print(f"✅ Generated {len(df)} track points")
print(f"\n📊 Platforms: {df['Platform'].unique().tolist()}")
print(f"📊 Tail Numbers: {df['Tail_Number'].unique().tolist()}")
df.head(10)

## 📊 4. Extract Analytical Insights

In [ ]:
# 1. Fleet Composition
fleet_stats = df.groupby('Platform')['Tail_Number'].nunique().reset_index(name='Active_Aircraft')
print("🛩️ FLEET COMPOSITION")
print("=" * 40)
print(fleet_stats.to_string(index=False))

# 2. Time on Station (ToS) per aircraft
tos_stats = df.groupby(['Platform', 'Tail_Number']).agg(
    Entry_Time=('Timestamp', 'min'),
    Exit_Time=('Timestamp', 'max'),
    Points=('Timestamp', 'count'),
    Avg_Altitude=('Altitude_ft', 'mean'),
    Avg_Speed=('Speed_kts', 'mean'),
).reset_index()

tos_stats['Time_on_Station_Hours'] = (tos_stats['Exit_Time'] - tos_stats['Entry_Time']).dt.total_seconds() / 3600

print("\n⏱️ TIME ON STATION")
print("=" * 40)
print(tos_stats[['Platform', 'Tail_Number', 'Time_on_Station_Hours', 'Avg_Altitude']].to_string(index=False))

# 3. Coverage statistics
print("\n📍 COVERAGE AREA")
print("=" * 40)
print(f"Latitude Range: {df['Latitude'].min():.2f}° to {df['Latitude'].max():.2f}°")
print(f"Longitude Range: {df['Longitude'].min():.2f}° to {df['Longitude'].max():.2f}°")
print(f"Altitude Range: {df['Altitude_ft'].min():,} to {df['Altitude_ft'].max():,} ft")

## 🔍 5. DBSCAN Clustering - Detect Loiter Zones

Automatically identify holding patterns and areas of concentrated ISR activity.

In [ ]:
# Cluster detection using DBSCAN
coords = df[['Latitude', 'Longitude']].values
clustering = DBSCAN(eps=0.3, min_samples=10).fit(coords)
df['Cluster'] = clustering.labels_

# Identify loiter zones (clusters)
loiter_zones = df[df['Cluster'] >= 0].groupby('Cluster').agg(
    Center_Lat=('Latitude', 'mean'),
    Center_Lon=('Longitude', 'mean'),
    Points=('Cluster', 'count'),
    Platforms=('Platform', lambda x: list(x.unique())),
).reset_index()

print("🎯 DETECTED LOITER ZONES")
print("=" * 50)
for _, zone in loiter_zones.iterrows():
    print(f"Zone {zone['Cluster']}: ({zone['Center_Lat']:.3f}, {zone['Center_Lon']:.3f})")
    print(f"   Points: {zone['Points']} | Platforms: {zone['Platforms']}")

## 🗺️ 6. Interactive Map Visualization

In [ ]:
# Create base map
m = folium.Map(
    location=[REGION['center_lat'], REGION['center_lon']],
    zoom_start=6,
    tiles='CartoDB dark_matter',  # Dark tactical style
)

# Color coding for platforms
colors = {
    'MQ-9': '#ff4444',      # Red - Strike/Recon
    'RC-135': '#44ff44',    # Green - SIGINT
    'RQ-4': '#4444ff',      # Blue - High-alt surveillance
    'P-8': '#ffff44',       # Yellow - Maritime
}

# Add strategic points
for name, (lat, lon) in STRATEGIC_POINTS.items():
    icon_color = 'red' if 'Nuclear' in name else ('blue' if 'US' in name or 'NAVCENT' in name else 'gray')
    folium.Marker(
        [lat, lon],
        popup=f"<b>{name}</b>",
        icon=folium.Icon(color=icon_color, icon='star' if 'Nuclear' in name else 'flag'),
    ).add_to(m)

# Add flight tracks
for tail in df['Tail_Number'].unique():
    track_df = df[df['Tail_Number'] == tail].sort_values('Timestamp')
    platform = track_df['Platform'].iloc[0]
    
    # Create track line
    points = track_df[['Latitude', 'Longitude']].values.tolist()
    
    folium.PolyLine(
        points,
        weight=2,
        color=colors.get(platform, '#ffffff'),
        opacity=0.7,
        popup=f"{platform} ({tail})",
    ).add_to(m)
    
    # Add current position marker
    last = track_df.iloc[-1]
    folium.CircleMarker(
        [last['Latitude'], last['Longitude']],
        radius=6,
        color=colors.get(platform, '#ffffff'),
        fill=True,
        fillOpacity=0.8,
        popup=f"<b>{platform} {tail}</b><br>Alt: {last['Altitude_ft']:,} ft<br>Speed: {last['Speed_kts']} kts",
    ).add_to(m)

# Add loiter zone circles
for _, zone in loiter_zones.iterrows():
    folium.Circle(
        [zone['Center_Lat'], zone['Center_Lon']],
        radius=30000,  # 30km radius
        color='#ff00ff',
        fill=True,
        fillOpacity=0.1,
        popup=f"Loiter Zone {zone['Cluster']}",
    ).add_to(m)

# Add legend
legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; 
            background: rgba(0,0,0,0.8); padding: 10px; border-radius: 5px;">
    <p style="color: white; margin: 0 0 5px 0;"><b>ISR Platforms</b></p>
    <p style="color: #ff4444; margin: 0;">● MQ-9 Reaper</p>
    <p style="color: #44ff44; margin: 0;">● RC-135 Rivet Joint</p>
    <p style="color: #4444ff; margin: 0;">● RQ-4 Global Hawk</p>
    <p style="color: #ffff44; margin: 0;">● P-8 Poseidon</p>
    <p style="color: #ff00ff; margin: 5px 0 0 0;">◯ Loiter Zones</p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

print("🗺️ Interactive map created!")
m

## 🔥 7. Heatmap - ISR Activity Density

In [ ]:
# Create heatmap
m_heat = folium.Map(
    location=[REGION['center_lat'], REGION['center_lon']],
    zoom_start=6,
    tiles='CartoDB dark_matter',
)

# Prepare heat data
heat_data = df[['Latitude', 'Longitude']].values.tolist()

HeatMap(
    heat_data,
    radius=15,
    blur=10,
    max_zoom=10,
    gradient={0.4: 'blue', 0.6: 'lime', 0.8: 'yellow', 1.0: 'red'},
).add_to(m_heat)

# Add strategic points
for name, (lat, lon) in STRATEGIC_POINTS.items():
    if 'Nuclear' in name:
        folium.Marker([lat, lon], popup=name, icon=folium.Icon(color='red', icon='warning')).add_to(m_heat)

print("🔥 Heatmap showing ISR concentration areas")
m_heat

## 📈 8. Plotly Analytics Dashboard

In [ ]:
# Altitude over time by platform
fig = px.scatter(
    df,
    x='Timestamp',
    y='Altitude_ft',
    color='Platform',
    hover_data=['Tail_Number', 'Speed_kts', 'Pattern'],
    title='ISR Platform Altitude Profiles',
    color_discrete_map=colors,
)
fig.update_layout(template='plotly_dark', height=400)
fig.show()

# Time on Station comparison
fig2 = px.bar(
    tos_stats,
    x='Tail_Number',
    y='Time_on_Station_Hours',
    color='Platform',
    title='Time on Station by Aircraft',
    color_discrete_map=colors,
)
fig2.update_layout(template='plotly_dark', height=400)
fig2.show()

## 💾 9. Export Data

In [ ]:
# Convert to GeoDataFrame
geometry = [Point(xy) for xy in zip(df['Longitude'], df['Latitude'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

# Export to various formats
# 1. CSV
df.to_csv('isr_tracks.csv', index=False)
print('✅ Saved: isr_tracks.csv')

# 2. GeoJSON
gdf_export = gdf.copy()
gdf_export['Timestamp'] = gdf_export['Timestamp'].astype(str)
gdf_export.to_file('isr_tracks.geojson', driver='GeoJSON')
print('✅ Saved: isr_tracks.geojson')

# 3. KML for Google Earth
kml = simplekml.Kml()
for tail in df['Tail_Number'].unique():
    track_df = df[df['Tail_Number'] == tail].sort_values('Timestamp')
    platform = track_df['Platform'].iloc[0]
    
    folder = kml.newfolder(name=f"{platform} - {tail}")
    
    # Add track line
    coords = [(row['Longitude'], row['Latitude'], row['Altitude_ft'] * 0.3048)  # Convert to meters
              for _, row in track_df.iterrows()]
    line = folder.newlinestring(name=f"{tail} Track")
    line.coords = coords
    line.altitudemode = simplekml.AltitudeMode.absolute
    line.style.linestyle.color = simplekml.Color.red if platform == 'MQ-9' else simplekml.Color.green
    line.style.linestyle.width = 2

kml.save('isr_tracks.kml')
print('✅ Saved: isr_tracks.kml')

# 4. Save analytics
tos_stats.to_csv('tos_analytics.csv', index=False)
print('✅ Saved: tos_analytics.csv')

# Save map as HTML
m.save('isr_map.html')
print('✅ Saved: isr_map.html')

In [ ]:
# Download all files
print("📥 Downloading files...")
files.download('isr_tracks.csv')
files.download('isr_tracks.geojson')
files.download('isr_tracks.kml')
files.download('tos_analytics.csv')
files.download('isr_map.html')

## 🌐 10. Fetch Real ADS-B Data (Optional)

Connect to OpenSky Network for live aircraft data.

In [ ]:
def fetch_opensky_data(bbox):
    """Fetch live aircraft from OpenSky Network API."""
    import requests
    
    lamin, lomin, lamax, lomax = bbox
    url = f"https://opensky-network.org/api/states/all?lamin={lamin}&lomin={lomin}&lamax={lamax}&lomax={lomax}"
    
    try:
        response = requests.get(url, timeout=30)
        if response.status_code == 200:
            data = response.json()
            states = data.get('states', [])
            
            aircraft = []
            for s in states:
                if s[5] and s[6]:  # Has lat/lon
                    aircraft.append({
                        'icao24': s[0],
                        'callsign': (s[1] or '').strip(),
                        'country': s[2],
                        'longitude': s[5],
                        'latitude': s[6],
                        'altitude_m': s[7] or 0,
                        'velocity_ms': s[9] or 0,
                        'heading': s[10] or 0,
                    })
            return pd.DataFrame(aircraft)
        else:
            print(f"API Error: {response.status_code}")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

# Fetch real data for Iran region
# bbox = (24.0, 44.0, 40.0, 64.0)  # lamin, lomin, lamax, lomax
# live_df = fetch_opensky_data(bbox)
# print(f"Fetched {len(live_df)} aircraft")
# live_df.head()

print("💡 Uncomment above to fetch LIVE aircraft data!")

---
## 📚 Resources

- **OpenSky Network**: https://opensky-network.org/
- **ADS-B Exchange**: https://adsbexchange.com/
- **CesiumJS** (3D Globe): https://cesium.com/
- **GDELT** (News/Events): https://gdeltproject.org/

---
Built with ✨ by Cleo | March 2026